<a href="https://colab.research.google.com/github/tnkkzk0322/keiba-horse-profile-extractor/blob/main/%E7%AB%B6%E9%A6%AC%E3%83%87%E3%83%BC%E3%82%BF%E6%8A%BD%E5%87%BA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ==============================
# 実行ごとに変更する設定
# ==============================
RACE_ID_LIST = [
    "202603020211",
    "202602010611",
]

OUTPUT_DIR = "/content/drive/MyDrive/netkeiba_output"

# 過去10年傾向として取得する過去レース件数。
# 各馬の過去成績を10走に制限する設定ではありません。
PAST_RACE_LIMIT = 10

# 通信間隔・タイムアウト
REQUEST_SLEEP_MIN = 5
REQUEST_SLEEP_MAX = 10
REQUEST_TIMEOUT_SEC = 30

WRITE_MATCHUP_CSV = False  # True にすると対戦表CSVも出力

# None の場合は対象レース名を使う。任意の検索語に変えたい場合だけ文字列を入れる。
PAST_RACE_SEARCH_WORD = None

import os
import time
import gzip
import atexit
import sqlite3
import hashlib
from collections import Counter
from typing import Optional, Dict, Set, List

# ------------------------------
# Colab / Drive 判定
# ------------------------------
try:
    from google.colab import drive as gdrive
    IN_COLAB = True
except Exception:
    gdrive = None
    IN_COLAB = False

# ------------------------------
# 設定
# ------------------------------
CACHE_NAMESPACE = "netkeiba_html_cache_v1"

# Colab ローカル
LOCAL_CACHE_ROOT = f"/content/{CACHE_NAMESPACE}"

# Google Drive 永続化先
DRIVE_MOUNT_POINT = "/content/drive"
DRIVE_CACHE_ROOT = f"{DRIVE_MOUNT_POINT}/MyDrive/{CACHE_NAMESPACE}"

# シャード数
# 16 か 32 を推奨。まずは 16 で十分。
CACHE_SHARD_COUNT = 16

# メモリキャッシュ
ENABLE_MEMORY_CACHE = True
HTML_CACHE: Dict[str, str] = {}

CACHE_METRICS = Counter()


def get_cache_metrics() -> Dict[str, int]:
    return dict(CACHE_METRICS)


def print_cache_metrics(label: str = "CACHE METRICS"):
    d = dict(CACHE_METRICS)
    print(f"========== {label} ==========")
    print(f"memory_hit    : {d.get('memory_hit', 0)}")
    print(f"sqlite_hit    : {d.get('sqlite_hit', 0)}")
    print(f"network_fetch : {d.get('network_fetch', 0)}")
    print(f"cache_bypass  : {d.get('cache_bypass', 0)}")
    print(f"drive_restore : {d.get('drive_restore', 0)}")
    print(f"sqlite_miss   : {d.get('sqlite_miss', 0)}")
    print(f"mojibake_repair: {d.get('mojibake_repair', 0)}")
    print("=================================")

# SQLite / gzip / 同期設定
CACHE_COMPRESSLEVEL = 5
SQLITE_TIMEOUT_SEC = 60
CACHE_SYNC_EVERY_N_WRITES = 100     # 100件ごとに Drive 同期
CACHE_SYNC_INTERVAL_SEC = 300       # 5分ごとに Drive 同期
CACHE_PRINT_LOG = False              # ログ出力したい場合 True


def cache_log(*args):
    if CACHE_PRINT_LOG:
        print("[CACHE]", *args)


class SQLiteShardHTMLCache:
    """
    URL -> HTML を SQLite に保存するキャッシュ
    - Colab ローカルに作業用 DB
    - Google Drive に永続化用 DB
    - URL ハッシュでシャード分割
    - 起動時に全コピーせず、必要シャードだけ Drive -> Local
    """

    def __init__(
        self,
        local_root: str,
        drive_root: str,
        shard_count: int = 16,
        compresslevel: int = 5,
        sync_every_n_writes: int = 100,
        sync_interval_sec: int = 300,
        sqlite_timeout_sec: int = 60,
        use_drive: bool = True,
    ):
        self.local_root = local_root
        self.drive_root = drive_root
        self.shard_count = shard_count
        self.compresslevel = compresslevel
        self.sync_every_n_writes = sync_every_n_writes
        self.sync_interval_sec = sync_interval_sec
        self.sqlite_timeout_sec = sqlite_timeout_sec
        self.use_drive = use_drive and IN_COLAB

        self._drive_ready = False
        self._local_loaded_shards: Set[int] = set()
        self._local_conns: Dict[int, sqlite3.Connection] = {}
        self._dirty_shards: Set[int] = set()

        self._writes_since_sync = 0
        self._last_sync_at = time.time()

        self._prepare()

    # --------------------------
    # 初期化
    # --------------------------
    def _prepare(self):
        os.makedirs(self.local_root, exist_ok=True)
        if self.use_drive:
            self._mount_drive_if_needed()
            os.makedirs(self.drive_root, exist_ok=True)

    def _mount_drive_if_needed(self):
        if not self.use_drive or self._drive_ready:
            return

        mydrive_path = os.path.join(DRIVE_MOUNT_POINT, "MyDrive")
        if not os.path.exists(mydrive_path):
            cache_log("Google Drive をマウントします")
            gdrive.mount(DRIVE_MOUNT_POINT, force_remount=False)

        os.makedirs(self.drive_root, exist_ok=True)
        self._drive_ready = True

    # --------------------------
    # パス / ハッシュ
    # --------------------------
    def _url_hash(self, url: str) -> str:
        return hashlib.sha1(url.encode("utf-8")).hexdigest()

    def _shard_id_from_url(self, url: str) -> int:
        return int(self._url_hash(url)[:8], 16) % self.shard_count

    def _local_db_path(self, shard_id: int) -> str:
        return os.path.join(self.local_root, f"cache_{shard_id:02d}.sqlite")

    def _drive_db_path(self, shard_id: int) -> str:
        return os.path.join(self.drive_root, f"cache_{shard_id:02d}.sqlite")

    # --------------------------
    # SQLite 基本
    # --------------------------
    def _connect(self, db_path: str) -> sqlite3.Connection:
        conn = sqlite3.connect(
            db_path,
            timeout=self.sqlite_timeout_sec,
            check_same_thread=False,
        )
        conn.row_factory = sqlite3.Row
        self._apply_pragmas(conn)
        self._init_schema(conn)
        return conn

    def _apply_pragmas(self, conn: sqlite3.Connection):
        conn.execute("PRAGMA journal_mode=DELETE;")
        conn.execute("PRAGMA synchronous=NORMAL;")
        conn.execute("PRAGMA temp_store=MEMORY;")
        conn.execute("PRAGMA foreign_keys=ON;")

    def _init_schema(self, conn: sqlite3.Connection):
        conn.execute("""
        CREATE TABLE IF NOT EXISTS html_cache (
            url TEXT PRIMARY KEY,
            url_hash TEXT NOT NULL,
            html_gzip BLOB NOT NULL,
            storage_encoding TEXT NOT NULL,
            fetched_at INTEGER NOT NULL,
            updated_at INTEGER NOT NULL,
            html_size INTEGER NOT NULL,
            compressed_size INTEGER NOT NULL
        )
        """)
        conn.execute("""
        CREATE INDEX IF NOT EXISTS idx_html_cache_url_hash
        ON html_cache(url_hash)
        """)
        conn.execute("""
        CREATE INDEX IF NOT EXISTS idx_html_cache_updated_at
        ON html_cache(updated_at)
        """)
        conn.commit()

    # --------------------------
    # Local shard 準備
    # --------------------------
    def _ensure_local_shard_loaded(self, shard_id: int):
        if shard_id in self._local_loaded_shards:
            return

        local_path = self._local_db_path(shard_id)
        drive_path = self._drive_db_path(shard_id)

        # Local DB が無ければ、Drive DB から必要シャードだけ復元
        if not os.path.exists(local_path):
            if self.use_drive and os.path.exists(drive_path):
                cache_log(f"Drive -> Local: shard {shard_id:02d} を読み込み")
                CACHE_METRICS["drive_restore"] += 1
                try:
                    src = sqlite3.connect(f"file:{drive_path}?mode=ro", uri=True, timeout=self.sqlite_timeout_sec)
                    dst = self._connect(local_path)
                    src.backup(dst)
                    dst.commit()
                    src.close()
                    dst.close()
                except Exception as e:
                    cache_log(f"Drive shard 読み込み失敗: shard={shard_id:02d}, error={e}")
                    if os.path.exists(local_path):
                        try:
                            os.remove(local_path)
                        except Exception:
                            pass
                    # 壊れていても空DBを作る
                    dst = self._connect(local_path)
                    dst.close()
            else:
                # 新規空DB
                dst = self._connect(local_path)
                dst.close()

        conn = self._connect(local_path)
        self._local_conns[shard_id] = conn
        self._local_loaded_shards.add(shard_id)

    def _get_local_conn(self, shard_id: int) -> sqlite3.Connection:
        self._ensure_local_shard_loaded(shard_id)
        return self._local_conns[shard_id]

    # --------------------------
    # 取得 / 保存
    # --------------------------
    def get(self, url: str) -> Optional[str]:
        shard_id = self._shard_id_from_url(url)
        conn = self._get_local_conn(shard_id)

        row = conn.execute(
            "SELECT html_gzip, storage_encoding FROM html_cache WHERE url = ?",
            (url,)
        ).fetchone()

        if row is None:
            return None

        try:
            html_bytes = gzip.decompress(row["html_gzip"])
            html = html_bytes.decode(row["storage_encoding"], errors="replace")
            return html
        except Exception as e:
            cache_log(f"gzip/decode 失敗: {url}, error={e}")
            return None

    def put_local(self, url: str, html: str):
        shard_id = self._shard_id_from_url(url)
        conn = self._get_local_conn(shard_id)

        now_ts = int(time.time())
        raw_bytes = html.encode("utf-8", errors="replace")
        gz_bytes = gzip.compress(raw_bytes, compresslevel=self.compresslevel)
        url_hash = self._url_hash(url)

        conn.execute("""
        INSERT INTO html_cache (
            url, url_hash, html_gzip, storage_encoding,
            fetched_at, updated_at, html_size, compressed_size
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(url) DO UPDATE SET
            url_hash = excluded.url_hash,
            html_gzip = excluded.html_gzip,
            storage_encoding = excluded.storage_encoding,
            updated_at = excluded.updated_at,
            html_size = excluded.html_size,
            compressed_size = excluded.compressed_size
        """, (
            url,
            url_hash,
            gz_bytes,
            "utf-8",
            now_ts,
            now_ts,
            len(raw_bytes),
            len(gz_bytes),
        ))
        conn.commit()

        self._dirty_shards.add(shard_id)
        self._writes_since_sync += 1

    # --------------------------
    # Drive 同期
    # --------------------------
    def sync_shard_to_drive(self, shard_id: int):
        if not self.use_drive:
            return
        if shard_id not in self._local_loaded_shards:
            return

        self._mount_drive_if_needed()

        local_conn = self._get_local_conn(shard_id)
        local_conn.commit()

        drive_path = self._drive_db_path(shard_id)

        try:
            dst = self._connect(drive_path)
            local_conn.backup(dst)
            dst.commit()
            dst.close()

            self._dirty_shards.discard(shard_id)
            cache_log(f"Local -> Drive: shard {shard_id:02d} を同期")
        except Exception as e:
            cache_log(f"Drive 同期失敗: shard={shard_id:02d}, error={e}")

    def sync_url_shard_to_drive(self, url: str):
        shard_id = self._shard_id_from_url(url)
        self.sync_shard_to_drive(shard_id)

    def sync_dirty_to_drive(self, force: bool = False):
        should_sync = (
            force
            or self._writes_since_sync >= self.sync_every_n_writes
            or (time.time() - self._last_sync_at) >= self.sync_interval_sec
        )

        if not should_sync:
            return

        dirty_shards = sorted(self._dirty_shards)
        if dirty_shards:
            cache_log(f"Drive 同期開始: dirty_shards={dirty_shards}")

        for shard_id in dirty_shards:
            self.sync_shard_to_drive(shard_id)

        self._writes_since_sync = 0
        self._last_sync_at = time.time()

    # --------------------------
    # 補助
    # --------------------------
    def count_local_records(self) -> int:
        total = 0
        for shard_id in range(self.shard_count):
            self._ensure_local_shard_loaded(shard_id)
            conn = self._get_local_conn(shard_id)
            row = conn.execute("SELECT COUNT(*) AS c FROM html_cache").fetchone()
            total += int(row["c"])
        return total

    def print_stats(self):
        total = 0
        loaded = sorted(self._local_loaded_shards)
        print("========== CACHE STATS ==========")
        print(f"shard_count          : {self.shard_count}")
        print(f"loaded_local_shards  : {loaded}")
        print(f"dirty_shards         : {sorted(self._dirty_shards)}")
        for shard_id in loaded:
            conn = self._get_local_conn(shard_id)
            row = conn.execute("SELECT COUNT(*) AS c FROM html_cache").fetchone()
            c = int(row["c"])
            total += c
            print(f"  shard {shard_id:02d}: {c} records")
        print(f"total_records        : {total}")
        print("=================================")

    def close(self):
        try:
            self.sync_dirty_to_drive(force=True)
        finally:
            for conn in self._local_conns.values():
                try:
                    conn.commit()
                    conn.close()
                except Exception:
                    pass
            self._local_conns.clear()
            self._local_loaded_shards.clear()


# ------------------------------
# インスタンス生成
# ------------------------------
CACHE = SQLiteShardHTMLCache(
    local_root=LOCAL_CACHE_ROOT,
    drive_root=DRIVE_CACHE_ROOT,
    shard_count=CACHE_SHARD_COUNT,
    compresslevel=CACHE_COMPRESSLEVEL,
    sync_every_n_writes=CACHE_SYNC_EVERY_N_WRITES,
    sync_interval_sec=CACHE_SYNC_INTERVAL_SEC,
    sqlite_timeout_sec=SQLITE_TIMEOUT_SEC,
    use_drive=True,
)


# ------------------------------
# 既存 fetch_soup 互換関数
# ------------------------------
def load_html_from_local_disk_cache(url: str) -> Optional[str]:
    """
    既存コード互換:
    Colab ローカルを見るように見せかけつつ、
    必要なら Drive から該当 shard を自動ロードして返す。
    """
    return CACHE.get(url)


def save_html_to_local_disk_cache(url: str, html: str):
    """
    既存コード互換:
    ローカル作業DBへ保存。
    """
    CACHE.put_local(url, html)


def save_html_to_drive_backup_cache(url: str, html: Optional[str] = None):
    """
    既存コード互換:
    以前の per-file 即時保存はやめて、閾値/時間ベースでまとめて同期する。
    これで Drive 書き込み回数を抑える。
    """
    CACHE.sync_dirty_to_drive(force=False)


def sync_all_cache_to_drive():
    """
    最後に明示的に呼ぶ用。
    """
    CACHE.sync_dirty_to_drive(force=True)


def print_cache_stats():
    CACHE.print_stats()


def clear_memory_cache():
    HTML_CACHE.clear()


@atexit.register
def _close_cache_on_exit():
    try:
        CACHE.close()
    except Exception as e:
        cache_log(f"終了時同期で例外: {e}")

# ==============================
# netkeiba scraper
# ==============================

import requests
from bs4 import BeautifulSoup
import json
import re
import random
import csv
import unicodedata
from datetime import date
from urllib.parse import quote

# --------------------------------------------------
# キャッシュ鮮度ポリシー
# --------------------------------------------------
# 競走馬の成績一覧ページは、同じ馬を過去に取得済みでも、
# 直近で別レースに出走していると内容が増えている可能性がある。
# そのため horse/result ページだけはキャッシュを「読む」のを避け、
# 毎回ネットワークから取り直す。
# 取得できたHTMLはキャッシュへ「保存」するので、設定を戻した場合や
# デバッグ時には最新寄りのキャッシュが残る。
FORCE_REFRESH_HORSE_RESULT_PAGES = True
CACHE_HORSE_RESULT_PAGES_AFTER_REFRESH = True

# RACE_ID_LIST の各 race_id で最初に開く入口ページ（出馬表ページ）も、
# キャッシュを読まずに毎回ネットワークから取り直す。
# 複数race_idを処理する場合、「各レースの最初のページ」が古いキャッシュにならない。
FORCE_REFRESH_ENTRY_PAGE_PER_RACE = True
CACHE_ENTRY_PAGE_AFTER_REFRESH = True

# 競走馬ページを都度アクセスする場合でも、そこからリンクされる過去レース結果ページ
# や、過去10年傾向のレース結果ページは従来どおりキャッシュ利用する。

# 旧 Mozilla UA のみ、ではなく、デバッグで通った構成に合わせる
HEADERS = {
    "User-Agent": "MyResearchBot/1.0",
    "Accept-Language": "ja,en;q=0.8",
}

REMOVE_KEYS_PER_RECORD = [
    "映像", "オッズ", "人気", "水分量", "馬場指数", "ﾀｲﾑ指数", "厩舎ｺﾒﾝﾄ", "備考", "賞金"
]

JYO_NAME_TO_CODE = {
    "札幌": "01",
    "函館": "02",
    "福島": "03",
    "新潟": "04",
    "東京": "05",
    "中山": "06",
    "中京": "07",
    "京都": "08",
    "阪神": "09",
    "小倉": "10",
    "門別": "30",
    "盛岡": "35",
    "水沢": "36",
    "浦和": "42",
    "船橋": "43",
    "大井": "44",
    "川崎": "45",
    "金沢": "46",
    "笠松": "47",
    "名古屋": "48",
    "園田": "50",
    "姫路": "51",
    "高知": "54",
    "佐賀": "55",
    "帯広(ば)": "65",
}

GRADE_NAME_TO_CODE = {
    "G1": "1",
    "G2": "2",
    "G3": "3",
    "OP": "4",
    "3勝": "5",
    "2勝": "6",
    "L": "11",
}

# ==============================
# 通信まわり
# ==============================
SESSION = requests.Session()
SESSION.headers.update(HEADERS)
SESSION.trust_env = False  # Colab 環境変数由来の proxy 等を避ける

RETRYABLE_STATUS_CODES = {429, 500, 502, 503, 504}
MAX_FETCH_RETRIES = 2
DEBUG_FETCH = False
_last_request_at = 0.0


def fetch_log(*args):
    if DEBUG_FETCH:
        print("[FETCH]", *args)


def normalize_ws(s: str) -> str:
    if not s:
        return ""
    s = str(s).replace(" ", " ")
    s = re.sub(r"\s+", " ", s).strip()

    # netkeiba の EUC-JP ページが latin-1/cp125x としてキャッシュされた場合、
    # get_text() 後の短い文字列だけが「¤«¤·¤ï」「¥·¥ã」のように化ける。
    # ここで短い文字列/部分文字列にも修復をかける。
    try:
        s = repair_eucjp_mojibake_text(s)
    except NameError:
        # 関数定義順の都合で、モジュール初期化中は未定義の可能性がある。
        pass
    return s


def normalize_netkeiba_url(url: str) -> str:
    """db.netkeiba.com の主要URLを正規化"""
    url = str(url).strip()

    # query付きURLは末尾スラッシュを付けない。
    if "?" in url:
        return url

    if url.startswith("https://db.netkeiba.com/race/") and not url.endswith("/"):
        url += "/"
    elif url.startswith("https://db.netkeiba.com/horse/result/") and not url.endswith("/"):
        url += "/"
    elif url.startswith("https://db.netkeiba.com/horse/ped/") and not url.endswith("/"):
        url += "/"
    elif re.match(r"^https://db\.netkeiba\.com/horse/[^/]+$", url):
        url += "/"

    return url


def make_horse_result_url(horse_id: str) -> str:
    horse_id = str(horse_id).strip().strip("/")
    return f"https://db.netkeiba.com/horse/result/{horse_id}/"


def is_horse_result_url(url: str) -> bool:
    """競走馬の成績一覧ページかどうかを判定する。"""
    u = normalize_netkeiba_url(url)
    return u.startswith("https://db.netkeiba.com/horse/result/")


def should_bypass_cache_read(url: str, force_refresh: bool = False) -> bool:
    """
    キャッシュの読み取りを避けるべきURLかどうか。

    - force_refresh=True が明示された場合
    - 設定で FORCE_REFRESH_HORSE_RESULT_PAGES=True かつ horse/result ページの場合
    """
    if force_refresh:
        return True
    if FORCE_REFRESH_HORSE_RESULT_PAGES and is_horse_result_url(url):
        return True
    return False


def wait_for_polite_interval():
    """アクセスはリクエスト前に待つ"""
    global _last_request_at
    min_wait = random.uniform(REQUEST_SLEEP_MIN, REQUEST_SLEEP_MAX)
    now = time.monotonic()
    elapsed = now - _last_request_at
    if elapsed < min_wait:
        time.sleep(min_wait - elapsed)
    _last_request_at = time.monotonic()


MOJIBAKE_MARKERS = "¤¥¡¢£¦§¨©ª«¬®¯°±²³´µμ¶·¸¹º»¼½¾¿ÀÁÂÃÄÅÆÇÈÉÊËÌÍÎÏÐÑÒÓÔÕÖ×ØÙÚÛÜÝÞßàáâãäåæçèéêëìíîïðñòóôõö÷øùúûüýþÿÇãïŞşĞğİı"

# cp1254 で一度Unicode化された可能性がある文字を、元の1byte値へ戻すための表。
# 例: EUC-JP の 0xB5 が NFKC 等で Greek μ になると latin-1 に戻せないため、ここで救う。
MOJIBAKE_EXTRA_BYTE_MAP = {
    "μ": 0xB5,
    "Ş": 0xDE,
    "ş": 0xFE,
    "Ğ": 0xD0,
    "ğ": 0xF0,
    "İ": 0xDD,
    "ı": 0xFD,
}

MOJIBAKE_SEGMENT_RE = re.compile(
    r"[\u00A0-\u00FFμŞşĞğİı]{2,}"
)


def count_japanese_chars(s: str) -> int:
    if not s:
        return 0
    n = 0
    for ch in s:
        o = ord(ch)
        if (
            (0x3040 <= o <= 0x30FF)  # ひらがな・カタカナ
            or (0x3400 <= o <= 0x9FFF)  # CJK
            or (0xF900 <= o <= 0xFAFF)
        ):
            n += 1
    return n


def count_mojibake_markers(s: str) -> int:
    if not s:
        return 0
    return sum(1 for ch in s if ch in MOJIBAKE_MARKERS)


def looks_like_eucjp_mojibake(s: str) -> bool:
    """
    EUC-JP のHTML/文字列を latin-1/cp125x 系で誤デコードした文字化けを判定する。

    例:
    - ¤«¤·¤ïµ­Ç° -> かしわ記念
    - ¥·¥ã¥ª¥ë -> シャオル
    - ¥À1600m -> ダ1600m
    """
    if not s:
        return False

    sample = str(s)[:20000]
    marker_count = count_mojibake_markers(sample)
    if marker_count < 2:
        return False

    jp_count = count_japanese_chars(sample)

    # 短い抽出済み文字列は marker が2個以上あれば修復候補にする。
    if len(sample) <= 200:
        return marker_count >= 2

    # HTML全体など長い文字列では、既に日本語が十分ある場合は全体変換を避ける。
    return marker_count > max(8, jp_count * 2)


def _mojibake_string_to_bytes(seg: str, encoding: str) -> Optional[bytes]:
    """
    誤デコード済みの文字列を、元のEUC-JPバイト列らしきものへ戻す。
    latin-1 で戻せない Greek μ / Turkish 文字は明示マップで戻す。
    """
    raw = bytearray()
    for ch in seg:
        o = ord(ch)
        if o <= 0xFF:
            raw.append(o)
            continue
        mapped = MOJIBAKE_EXTRA_BYTE_MAP.get(ch)
        if mapped is not None:
            raw.append(mapped)
            continue
        try:
            encoded = ch.encode(encoding, errors="strict")
            if len(encoded) == 1:
                raw.extend(encoded)
                continue
        except Exception:
            pass
        return None
    return bytes(raw)


def _score_decoded_text(s: str) -> int:
    if not s:
        return -10**9
    sample = s[:20000]
    jp = count_japanese_chars(sample)
    marker = count_mojibake_markers(sample)
    replacement = sample.count("�")
    # ひらがな/カタカナ/漢字が増え、文字化け記号と置換文字が減るものを採用する。
    return jp * 4 - marker * 3 - replacement * 8


def _repair_one_mojibake_segment(seg: str) -> str:
    if not seg or count_mojibake_markers(seg) < 1:
        return seg

    before_score = _score_decoded_text(seg)
    best = seg
    best_score = before_score

    for wrong_encoding in ("latin-1", "cp1252", "cp1254"):
        raw = _mojibake_string_to_bytes(seg, wrong_encoding)
        if not raw:
            continue
        for right_encoding in ("euc-jp", "euc_jis_2004", "euc_jisx0213"):
            try:
                fixed = raw.decode(right_encoding, errors="replace")
            except Exception:
                continue
            score = _score_decoded_text(fixed)
            if score > best_score:
                best = fixed
                best_score = score

    return best


def repair_eucjp_mojibake_text(s: str) -> str:
    """
    EUC-JP bytes -> latin-1/cp125x decoded string になった文字列を復元する。

    前回版との違い:
    - HTML全体だけでなく、抽出後の短い文字列を修復する。
    - 「20:05発走 / ¥À1600m」のような混在文字列では、¥À だけを部分修復する。
    - NFKC等で µ が μ に変わった場合も 0xB5 として戻す。
    """
    if not isinstance(s, str) or not looks_like_eucjp_mojibake(s):
        return s

    # 文字列全体がほぼ mojibake の場合は全体変換を試す。
    whole_fixed = _repair_one_mojibake_segment(s)
    if whole_fixed != s and _score_decoded_text(whole_fixed) > _score_decoded_text(s):
        return whole_fixed

    # 日本語と mojibake が混在する場合は、怪しい連続部分だけを修復する。
    def repl(m: re.Match) -> str:
        seg = m.group(0)
        if count_mojibake_markers(seg) < 1:
            return seg
        fixed = _repair_one_mojibake_segment(seg)
        return fixed

    fixed = MOJIBAKE_SEGMENT_RE.sub(repl, s)
    return fixed


def decode_response_text(resp: requests.Response, url: str) -> str:
    """
    requests.Response を、netkeibaの文字コード事情に合わせて安全に文字列化する。
    res.text だけに頼ると EUC-JP ページで文字化けキャッシュが残ることがある。
    """
    apply_best_effort_encoding(resp, url)
    encoding = resp.encoding or "utf-8"
    try:
        html = resp.content.decode(encoding, errors="replace")
    except Exception:
        html = resp.text
    return repair_eucjp_mojibake_text(html)


def apply_best_effort_encoding(resp: requests.Response, url: str) -> requests.Response:
    """charset を優先し、無ければ HTML 先頭から判定。netkeibaのDB/NAR系は最後に EUC-JP を優先。"""
    content_type = (resp.headers.get("Content-Type") or "").lower()
    if "charset=euc-jp" in content_type:
        resp.encoding = "euc-jp"
        return resp
    if "charset=shift_jis" in content_type:
        resp.encoding = "shift_jis"
        return resp
    if "charset=utf-8" in content_type:
        resp.encoding = "utf-8"
        return resp

    head = resp.content[:3000].lower()
    if b"charset=euc-jp" in head:
        resp.encoding = "euc-jp"
    elif b"charset=shift_jis" in head:
        resp.encoding = "shift_jis"
    elif b"charset=utf-8" in head:
        resp.encoding = "utf-8"
    else:
        if "db.netkeiba.com" in url or "nar.netkeiba.com" in url:
            resp.encoding = "euc-jp"
        else:
            resp.encoding = resp.apparent_encoding or "utf-8"
    return resp


def debug_response(resp: requests.Response, label: str = ""):
    apply_best_effort_encoding(resp, resp.request.url)
    print("[FETCH]", "=" * 80)
    print("[FETCH]", label)
    print("[FETCH]", "request.method :", resp.request.method)
    print("[FETCH]", "request.url :", resp.request.url)
    print("[FETCH]", "status_code :", resp.status_code)
    print("[FETCH]", "final_url :", resp.url)
    print("[FETCH]", "history :", [(h.status_code, h.headers.get("Location")) for h in resp.history])
    print("[FETCH]", "server :", resp.headers.get("Server"))
    print("[FETCH]", "content_type :", resp.headers.get("Content-Type"))
    print("[FETCH]", "cookie(sent) :", resp.request.headers.get("Cookie"))
    print("[FETCH]", "body_head :", repr(resp.text[:200]))
    print("[FETCH]", "=" * 80)

def validate_fetched_html(url: str, html: str):
    """
    status_code=200 で返った異常ページをキャッシュへ保存しないための軽い検証。
    正常ページの構造変更で誤検知しにくいよう、明らかなブロック/エラー文言だけを見る。
    """
    if not html:
        raise RuntimeError(f"empty html: {url}")

    head_text = normalize_ws(html[:8000])
    blocked_markers = [
        "Access Denied",
        "Forbidden",
        "Service Temporarily Unavailable",
        "Too Many Requests",
        "アクセスが集中",
        "しばらく時間をおいて",
        "ただいまアクセスが集中",
    ]

    for marker in blocked_markers:
        if marker in head_text:
            raise RuntimeError(f"blocked or invalid page detected: {url} | marker={marker}")


def fetch_html(
    url: str,
    *,
    force_refresh: bool = False,
    read_cache: bool = True,
    write_cache: bool = True,
) -> str:
    """
    HTMLを取得する共通関数。

    通常: memory cache -> sqlite cache -> network の順。
    force_refresh=True または horse/result ページ設定に該当する場合:
      - キャッシュは読まず、必ずネットワークへ行く
      - 成功したHTMLは write_cache=True ならキャッシュに保存する

    これにより、競走馬の成績一覧ページだけは鮮度を優先し、
    そこから参照される過去レース結果ページなどは従来どおりキャッシュできる。
    """
    url = normalize_netkeiba_url(url)

    bypass_cache = should_bypass_cache_read(url, force_refresh=force_refresh)
    cache_read_enabled = bool(read_cache) and not bypass_cache

    if not cache_read_enabled:
        CACHE_METRICS["cache_bypass"] += 1

    if cache_read_enabled and ENABLE_MEMORY_CACHE and url in HTML_CACHE:
        CACHE_METRICS["memory_hit"] += 1
        html = HTML_CACHE[url]
        fixed = repair_eucjp_mojibake_text(html)
        if fixed != html:
            CACHE_METRICS["mojibake_repair"] += 1
            HTML_CACHE[url] = fixed
            if write_cache:
                save_html_to_local_disk_cache(url, fixed)
                save_html_to_drive_backup_cache(url, fixed)
        return fixed

    if cache_read_enabled:
        local_html = load_html_from_local_disk_cache(url)
        if local_html is not None:
            CACHE_METRICS["sqlite_hit"] += 1
            fixed = repair_eucjp_mojibake_text(local_html)
            if fixed != local_html:
                CACHE_METRICS["mojibake_repair"] += 1
                if write_cache:
                    save_html_to_local_disk_cache(url, fixed)
                    save_html_to_drive_backup_cache(url, fixed)
            if ENABLE_MEMORY_CACHE:
                HTML_CACHE[url] = fixed
            return fixed

        CACHE_METRICS["sqlite_miss"] += 1

    last_exc = None

    for attempt in range(MAX_FETCH_RETRIES + 1):
        try:
            wait_for_polite_interval()
            CACHE_METRICS["network_fetch"] += 1
            res = SESSION.get(url, timeout=REQUEST_TIMEOUT_SEC, allow_redirects=True)
            html_text = decode_response_text(res, url)

            if DEBUG_FETCH:
                debug_response(res, f"attempt={attempt} url={url}")

            if res.status_code == 200:
                validate_fetched_html(url, html_text)
                html = html_text
                if ENABLE_MEMORY_CACHE:
                    HTML_CACHE[url] = html
                if write_cache:
                    save_html_to_local_disk_cache(url, html)
                    save_html_to_drive_backup_cache(url, html)
                return html

            if res.status_code == 403:
                body_head = html_text[:200].replace("\n", " ")
                raise requests.HTTPError(
                    f"403 Forbidden for url: {url} | body_head={body_head!r}",
                    response=res,
                )

            if res.status_code in RETRYABLE_STATUS_CODES and attempt < MAX_FETCH_RETRIES:
                backoff = (2 ** attempt) + random.uniform(0.3, 0.8)
                fetch_log(f"retryable status={res.status_code}, sleep={backoff:.2f}s, url={url}")
                time.sleep(backoff)
                continue

            res.raise_for_status()

        except requests.HTTPError as e:
            last_exc = e
            status = getattr(getattr(e, "response", None), "status_code", None)
            if status == 403:
                raise
            if attempt < MAX_FETCH_RETRIES:
                backoff = (2 ** attempt) + random.uniform(0.3, 0.8)
                fetch_log(f"http retry sleep={backoff:.2f}s, error={repr(e)}")
                time.sleep(backoff)
                continue
            raise

        except (requests.ConnectionError, requests.Timeout) as e:
            last_exc = e
            if attempt < MAX_FETCH_RETRIES:
                backoff = (2 ** attempt) + random.uniform(0.3, 0.8)
                fetch_log(f"network retry sleep={backoff:.2f}s, error={repr(e)}")
                time.sleep(backoff)
                continue
            raise

        except Exception as e:
            last_exc = e
            raise

    if last_exc is not None:
        raise last_exc
    raise RuntimeError(f"fetch_html failed: {url}")


def fetch_soup(
    url: str,
    *,
    force_refresh: bool = False,
    read_cache: bool = True,
    write_cache: bool = True,
) -> BeautifulSoup:
    html = fetch_html(
        url,
        force_refresh=force_refresh,
        read_cache=read_cache,
        write_cache=write_cache,
    )
    return BeautifulSoup(html, "html.parser")




# ==============================
# 血統情報取得
# ==============================
# 出馬表の各馬に以下を付与する。
# - 五代血統表: table.blood_table.detail を列ごとに上から順番で固定割当
# - クロス: div.blood_cross table
# - 兄弟: div.brother table
# - 叔父叔母: div.uncle table
#
# ステータスやエラー情報は JSON には入れず、既存方針どおり print のみ。
# HTML取得は既存 fetch_soup を使うため、SQLite/Driveキャッシュにも保存される。

HORSE_LINK_RE = re.compile(
    r"(?:https?://db\.netkeiba\.com)?/horse/(?:ped/|result/|sire/|mare/)?([0-9A-Za-z]+)/"
)


def empty_pedigree_info() -> Dict:
    return {
        "五代血統表": [],
        "クロス": [],
        "兄弟": [],
        "叔父叔母": [],
    }


def make_horse_ped_url(horse_id: str) -> str:
    horse_id = str(horse_id).strip().strip("/")
    return f"https://db.netkeiba.com/horse/ped/{horse_id}/"


def extract_horse_id_from_href(href: str) -> str:
    if not href:
        return ""
    m = HORSE_LINK_RE.search(str(href))
    return m.group(1) if m else ""


def clean_horse_link_name(raw: str) -> str:
    s = normalize_ws(raw)
    if not s:
        return ""

    s = re.sub(r"の(血統|戦績|プロフィール|競走馬データ|競走馬情報).*$", "", s)
    s = re.sub(r"\s*\|\s*.*$", "", s)
    s = re.sub(r"\s*\[[^\]]+\]\s*$", "", s)
    s = re.sub(r"\s+\d+(?:\.\d+)?%\s*$", "", s)
    return normalize_ws(s)


def horse_name_from_anchor(a) -> str:
    if a is None:
        return ""

    # 血統表では1つのa内に「日本語名<br>英名」が入ることがある。
    # get_text全体では混ざるため、最初の文字列だけを馬名として使う。
    strings = [normalize_ws(x) for x in a.stripped_strings if normalize_ws(x)]
    if strings:
        name = clean_horse_link_name(strings[0])
        if name:
            return name

    title = a.get("title", "")
    if title:
        name = clean_horse_link_name(title)
        if name:
            return name

    return ""


def horse_record_from_anchor(a, relation: str, extra: Optional[Dict] = None) -> Optional[Dict]:
    if a is None:
        return None

    horse_id = extract_horse_id_from_href(a.get("href", ""))
    horse_name = horse_name_from_anchor(a)

    if not horse_id and not horse_name:
        return None

    rec = {
        "間柄": relation,
        "馬名": horse_name,
        "馬ID": horse_id,
    }

    if extra:
        rec.update(extra)

    return rec


def dedupe_dict_list(records: List[Dict], key_fields: List[str]) -> List[Dict]:
    out = []
    seen = set()

    for r in records:
        key = tuple(r.get(k, "") for k in key_fields)
        if key in seen:
            continue
        seen.add(key)
        out.append(r)

    return out


# ------------------------------
# 五代血統表
# ------------------------------
def make_pedigree_relations(generation: int) -> List[str]:
    """
    generation=1 -> 父, 母
    generation=2 -> 父父, 父母, 母父, 母母
    generation=3 -> 父父父, 父父母, ...
    """
    if generation <= 0:
        return []

    out = []
    for i in range(2 ** generation):
        bits = format(i, f"0{generation}b")
        out.append("".join("父" if b == "0" else "母" for b in bits))
    return out


def select_five_generation_table(soup: BeautifulSoup):
    table = soup.select_one("table.blood_table.detail")
    if table is not None:
        return table

    table = soup.find("table", summary=re.compile("5代血統表"))
    if table is not None:
        return table

    # フォールバック: 馬リンクが多く、blood_tableっぽいものを選ぶ。
    best_table = None
    best_score = -1
    for t in soup.find_all("table"):
        cls = " ".join(t.get("class", []))
        horse_links = [
            a for a in t.find_all("a", href=True)
            if extract_horse_id_from_href(a.get("href", ""))
        ]
        if len(horse_links) < 10:
            continue
        score = len(horse_links)
        if re.search(r"blood|pedigree", cls, re.I):
            score += 100
        if score > best_score:
            best_score = score
            best_table = t
    return best_table


def build_table_cell_positions(table):
    """
    rowspan/colspanを展開し、各セルの開始row/colを得る。
    今回は世代推定にはrowspanを使わず、列番号だけを使う。
    """
    rows = table.find_all("tr")
    occupied = {}
    positions = {}

    for row_idx, tr in enumerate(rows):
        col_idx = 0
        cells = tr.find_all(["td", "th"], recursive=False)

        for cell in cells:
            while (row_idx, col_idx) in occupied:
                col_idx += 1

            try:
                rowspan = int(cell.get("rowspan", 1) or 1)
            except Exception:
                rowspan = 1

            try:
                colspan = int(cell.get("colspan", 1) or 1)
            except Exception:
                colspan = 1

            cell_id = id(cell)
            if cell_id not in positions:
                positions[cell_id] = {
                    "cell": cell,
                    "row": row_idx,
                    "col": col_idx,
                    "rowspan": rowspan,
                    "colspan": colspan,
                }

            for dr in range(rowspan):
                for dc in range(colspan):
                    occupied[(row_idx + dr, col_idx + dc)] = cell

            col_idx += colspan

    return list(positions.values())


def first_profile_anchor_in_cell(cell):
    """
    セル内の代表馬リンクを返す。
    [血統]/[産駒]リンクではなく、セル先頭の /horse/{id}/ を優先する。
    """
    for a in cell.find_all("a", href=True):
        href = str(a.get("href", ""))
        horse_id = extract_horse_id_from_href(href)
        if not horse_id:
            continue

        # /horse/ped/{id}/ や /horse/sire/{id}/ は補助リンクなので、通常プロフィールを優先。
        if re.search(r"/horse/[0-9A-Za-z]+/", href):
            return a

    # 念のため、プロフィール以外のhorseリンクでも馬名が取れるなら返す。
    for a in cell.find_all("a", href=True):
        if extract_horse_id_from_href(a.get("href", "")) and horse_name_from_anchor(a):
            return a

    return None


def parse_five_generation_pedigree(soup: BeautifulSoup, target_horse_id: str = "") -> List[Dict]:
    table = select_five_generation_table(soup)
    if table is None:
        return []

    cell_positions = build_table_cell_positions(table)
    by_col = {}

    for info in cell_positions:
        cell = info["cell"]
        a = first_profile_anchor_in_cell(cell)
        if a is None:
            continue

        horse_id = extract_horse_id_from_href(a.get("href", ""))
        horse_name = horse_name_from_anchor(a)
        if not horse_id and not horse_name:
            continue
        if target_horse_id and horse_id == str(target_horse_id):
            continue

        col = info["col"]
        by_col.setdefault(col, []).append({
            "row": info["row"],
            "anchor": a,
            "馬ID": horse_id,
            "馬名": horse_name,
        })

    if not by_col:
        return []

    records = []

    # netkeibaの5代血統表は、左から世代1,2,3,4,5。
    # 各列内では上から順に 父系 -> 母系 の固定順。
    for generation, col in enumerate(sorted(by_col.keys()), start=1):
        if generation > 5:
            break

        horses = sorted(by_col[col], key=lambda x: x["row"])
        relations = make_pedigree_relations(generation)

        # 余分な列や重複があった場合は、期待数までに制限する。
        for item, relation in zip(horses[:len(relations)], relations):
            rec = horse_record_from_anchor(
                item["anchor"],
                relation,
                extra={"世代": generation},
            )
            if rec:
                records.append(rec)

    return dedupe_dict_list(records, ["間柄", "馬ID", "馬名"])


# ------------------------------
# クロス
# ------------------------------
def parse_cross_info(soup: BeautifulSoup, five_generation_records: Optional[List[Dict]] = None) -> List[Dict]:
    records = []
    name_to_id = {}

    for r in five_generation_records or []:
        name = r.get("馬名", "")
        horse_id = r.get("馬ID", "")
        if name and horse_id and name not in name_to_id:
            name_to_id[name] = horse_id

    cross_box = soup.select_one("div.blood_cross")
    if cross_box is None:
        return []

    for tr in cross_box.select("table tr"):
        tds = tr.find_all("td")
        if len(tds) < 3:
            continue

        horse_name = clean_horse_link_name(tds[0].get_text(" ", strip=True))
        percent = normalize_ws(tds[1].get_text(" ", strip=True))
        cross_relation = normalize_ws(tds[2].get_text(" ", strip=True))

        if not horse_name or not cross_relation:
            continue

        a = tds[0].find("a", href=True)
        horse_id = extract_horse_id_from_href(a.get("href", "")) if a else ""
        if not horse_id:
            horse_id = name_to_id.get(horse_name, "")

        cross_text = normalize_ws(" ".join([x for x in [horse_name, percent, cross_relation] if x]))

        records.append({
            "間柄": cross_relation,
            "馬名": horse_name,
            "馬ID": horse_id,
            "クロス表記": cross_text,
        })

    return dedupe_dict_list(records, ["馬ID", "馬名", "間柄", "クロス表記"])


# ------------------------------
# 兄弟 / 叔父叔母
# ------------------------------
def birth_year_from_horse_id(horse_id: str) -> Optional[int]:
    s = str(horse_id or "")
    if re.match(r"^[12]\d{3}", s):
        return int(s[:4])
    return None


def extract_birth_year_from_text(text: str) -> Optional[int]:
    text = normalize_ws(text)
    m = re.search(r"([12]\d{3})年?", text)
    if m:
        return int(m.group(1))
    return None


def extract_target_birth_year(soup: Optional[BeautifulSoup], horse_id: str = "") -> Optional[int]:
    if soup is not None:
        text = normalize_ws(soup.get_text(" ", strip=True))
        m = re.search(r"生年月日\s*([12]\d{3})年", text)
        if m:
            return int(m.group(1))
        m = re.search(r"([12]\d{3})年\d{1,2}月\d{1,2}日", text)
        if m:
            return int(m.group(1))

    return birth_year_from_horse_id(horse_id)


def infer_sibling_relation(sex: str, birth_year: Optional[int], target_birth_year: Optional[int]) -> str:
    if birth_year is not None and target_birth_year is not None:
        if birth_year < target_birth_year:
            if sex in ("牡", "セ"):
                return "兄"
            if sex == "牝":
                return "姉"
            return "兄姉"
        if birth_year > target_birth_year:
            if sex in ("牡", "セ"):
                return "弟"
            if sex == "牝":
                return "妹"
            return "弟妹"

    return "兄弟"


def infer_uncle_aunt_relation(sex: str) -> str:
    if sex in ("牡", "セ"):
        return "叔父"
    if sex == "牝":
        return "叔母"
    return "叔父叔母"


def parse_relative_table_rows(table, target_horse_id: str = "") -> List[Dict]:
    rows = []
    if table is None:
        return rows

    for tr in table.find_all("tr"):
        tds = tr.find_all("td")
        if len(tds) < 3:
            continue

        a = tds[0].find("a", href=True)
        if a is None:
            continue

        horse_id = extract_horse_id_from_href(a.get("href", ""))
        if not horse_id:
            continue
        if target_horse_id and horse_id == str(target_horse_id):
            continue

        horse_name = horse_name_from_anchor(a)
        sex = normalize_ws(tds[1].get_text(" ", strip=True))
        birth_year = extract_birth_year_from_text(tds[2].get_text(" ", strip=True))

        rows.append({
            "馬名": horse_name,
            "馬ID": horse_id,
            "性": sex,
            "生年": birth_year,
        })

    return rows


def parse_siblings_from_ped_soup(
    soup: Optional[BeautifulSoup],
    target_horse_id: str = "",
    target_birth_year: Optional[int] = None,
) -> List[Dict]:
    if soup is None:
        return []

    table = soup.select_one("div.brother table")
    rows = parse_relative_table_rows(table, target_horse_id=target_horse_id)

    records = []
    for row in rows:
        relation = infer_sibling_relation(
            sex=row.get("性", ""),
            birth_year=row.get("生年"),
            target_birth_year=target_birth_year,
        )
        records.append({
            "間柄": relation,
            "馬名": row.get("馬名", ""),
            "馬ID": row.get("馬ID", ""),
        })

    return dedupe_dict_list(records, ["間柄", "馬ID", "馬名"])


def parse_uncle_aunt_from_ped_soup(
    soup: Optional[BeautifulSoup],
    target_horse_id: str = "",
) -> List[Dict]:
    if soup is None:
        return []

    table = soup.select_one("div.uncle table")
    rows = parse_relative_table_rows(table, target_horse_id=target_horse_id)

    records = []
    for row in rows:
        relation = infer_uncle_aunt_relation(row.get("性", ""))
        records.append({
            "間柄": relation,
            "馬名": row.get("馬名", ""),
            "馬ID": row.get("馬ID", ""),
        })

    return dedupe_dict_list(records, ["間柄", "馬ID", "馬名"])


# ------------------------------
# 統合
# ------------------------------
def fetch_horse_pedigree_info(horse_id: str) -> Dict:
    """馬IDから血統情報を取得。既存キャッシュ機構を利用する。"""
    if not horse_id:
        return empty_pedigree_info()

    info = empty_pedigree_info()

    try:
        ped_soup = fetch_soup(
            make_horse_ped_url(horse_id),
            force_refresh=False,
            read_cache=True,
            write_cache=True,
        )
    except Exception as e:
        print(f"ERROR pedigree page: horse_id={horse_id}, error={e}")
        return info

    target_birth_year = extract_target_birth_year(ped_soup, horse_id=horse_id)

    info["五代血統表"] = parse_five_generation_pedigree(
        ped_soup,
        target_horse_id=horse_id,
    )
    info["クロス"] = parse_cross_info(
        ped_soup,
        five_generation_records=info["五代血統表"],
    )
    info["兄弟"] = parse_siblings_from_ped_soup(
        ped_soup,
        target_horse_id=horse_id,
        target_birth_year=target_birth_year,
    )
    info["叔父叔母"] = parse_uncle_aunt_from_ped_soup(
        ped_soup,
        target_horse_id=horse_id,
    )

    return info


def enrich_horses_with_pedigree(race_json: Dict):
    for horse in race_json.get("horses", []):
        horse_name = horse.get("馬名", "")
        horse_id = horse.get("馬ID", "")

        print(f"Fetching horse pedigree: {horse_name}...")

        try:
            horse["血統情報"] = fetch_horse_pedigree_info(horse_id)
        except Exception as e:
            print("ERROR pedigree:", horse_name, e)
            horse["血統情報"] = empty_pedigree_info()

def date_from_yyyymmdd(yyyymmdd: str) -> Optional[date]:
    """
    20260505 -> date(2026, 5, 5)
    """
    if not yyyymmdd:
        return None

    s = str(yyyymmdd).strip()
    if not re.fullmatch(r"\d{8}", s):
        return None

    try:
        return date(int(s[:4]), int(s[4:6]), int(s[6:8]))
    except Exception:
        return None


def parse_date_any(text: str) -> Optional[date]:
    """
    日付文字列を date に変換する。

    対応:
    - 2026年5月5日
    - 2026/05/05
    - 2026-05-05
    - 2026.05.05
    - 20260505  ※文字列全体が8桁の場合のみ
    """
    if not text:
        return None

    s = normalize_ws(str(text))

    patterns = [
        r"(\d{4})年\s*(\d{1,2})月\s*(\d{1,2})日",
        r"(\d{4})/(\d{1,2})/(\d{1,2})",
        r"(\d{4})-(\d{1,2})-(\d{1,2})",
        r"(\d{4})\.(\d{1,2})\.(\d{1,2})",
    ]

    for p in patterns:
        m = re.search(p, s)
        if m:
            try:
                return date(int(m.group(1)), int(m.group(2)), int(m.group(3)))
            except Exception:
                return None

    if re.fullmatch(r"\d{8}", s):
        return date_from_yyyymmdd(s)

    return None

def date_to_ymd_slash(d: Optional[date]) -> str:
    return d.strftime("%Y/%m/%d") if d else ""


def normalize_date_for_sort(date_str: str):
    d = parse_date_any(date_str)
    if d is None:
        return (0, 0, 0)
    return (d.year, d.month, d.day)


def sanitize_filename(name: str) -> str:
    if not name:
        return "レース名不明"
    s = normalize_ws(name)
    s = re.sub(r'[\\/:*?"<>|]', "_", s)
    s = re.sub(r"\s+", "_", s)
    s = s.strip("._")
    return s if s else "レース名不明"


def safe_int(x):
    try:
        return int(str(x))
    except Exception:
        return None


def to_int_safe(x):
    try:
        return int(str(x))
    except Exception:
        return None


def time_to_seconds(time_str: str):
    if not time_str or time_str in ("-", "計不"):
        return None
    match = re.match(r"(\d+):(\d+\.\d+)", str(time_str))
    if not match:
        return None
    minutes = int(match.group(1))
    seconds = float(match.group(2))
    return minutes * 60 + seconds


def seconds_to_time(sec: float):
    m = int(sec // 60)
    s = sec % 60
    return f"{m}:{s:04.1f}"


def write_jsonl(records: List[Dict], output_path: str):
    with open(output_path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def is_nar_jyo_code(code: str) -> bool:
    try:
        return int(str(code)) >= 30
    except Exception:
        return False


def is_nar_race_id(race_id: str) -> bool:
    if not race_id or len(str(race_id)) < 6:
        return False
    jyo_code = str(race_id)[4:6]
    return is_nar_jyo_code(jyo_code)

def build_shutuba_url_from_race_id(race_id: str) -> str:
    base = "https://nar.netkeiba.com" if is_nar_race_id(race_id) else "https://race.netkeiba.com"
    return f"{base}/race/shutuba.html?race_id={race_id}"


def build_result_url_from_race_id(race_id: str, is_nar: bool) -> str:
    base = "https://nar.netkeiba.com" if is_nar else "https://race.netkeiba.com"
    return f"{base}/race/result.html?race_id={race_id}"


def get_race_year_text(race_date: str) -> str:
    d = parse_date_any(race_date)
    return str(d.year) if d else "0000"


def build_output_prefix(race_json: Dict) -> str:
    safe_race_name = sanitize_filename(race_json.get("race_name", ""))
    race_year = get_race_year_text(race_json.get("race_date", ""))
    race_id = str(race_json.get("race_id") or extract_race_id_from_url(race_json.get("race_url", "")) or "unknown")
    return f"{race_id}_{safe_race_name}{race_year}"


def clean_race_name_text(text: str) -> str:
    s = normalize_ws(text)
    s = re.sub(r"\s*出馬表\s*\|.*$", "", s)
    s = re.sub(r"\s*\|.*$", "", s)
    s = re.sub(r"\s*-\s*netkeiba.*$", "", s)
    s = re.sub(r"\s*[\(（]?\s*重賞\s*[\)）]?\s*", " ", s)
    return normalize_ws(s)


def normalize_race_name_for_search(text: str) -> str:
    s = normalize_ws(text)
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"\s+", " ", s).strip()
    if not s:
        return ""
    return s.split(" ", 1)[0].strip()

def build_search_word_candidates(race_name: str) -> List[str]:
    s = normalize_ws(race_name)
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"\s+", " ", s).strip()

    head = normalize_race_name_for_search(s)
    head_no_kyoso = re.sub(r"競走$", "", head).strip()

    candidates = []
    for x in [head, head_no_kyoso, s]:
        if x and x not in candidates:
            candidates.append(x)
    return candidates

def normalize_grade_name(text: str) -> str:
    if not text:
        return ""

    s = normalize_ws(text)
    s = unicodedata.normalize("NFKC", s).upper()

    s = s.replace("JPN1", "G1")
    s = s.replace("JPN2", "G2")
    s = s.replace("JPN3", "G3")

    if "G1" in s:
        return "G1"
    if "G2" in s:
        return "G2"
    if "G3" in s:
        return "G3"
    if "リステッド" in s or re.search(r'(?:^|[\s(（])L(?:[\s)）]|$)', s):
        return "L"
    if "OP" in s or "オープン" in s:
        return "OP"
    if "3勝" in s or "3勝クラス" in s or "1600万" in s:
        return "3勝"
    if "2勝" in s or "2勝クラス" in s or "1000万" in s:
        return "2勝"
    return ""

def extract_grade_code_from_target_race(soup: BeautifulSoup, race_name: str) -> str:
    texts = []

    for sel in [
        "h1.RaceName",
        "div.RaceName h1",
        "div.RaceName",
        "div.RaceData01",
        "div.RaceData02",
        "p.smalltxt",
        "title",
    ]:
        tag = soup.select_one(sel)
        if tag:
            txt = normalize_ws(tag.get_text(" ", strip=True))
            if txt:
                texts.append(txt)

    meta_desc = soup.find("meta", attrs={"name": "description"})
    if meta_desc and meta_desc.get("content"):
        texts.append(normalize_ws(meta_desc.get("content")))

    og_title = soup.find("meta", attrs={"property": "og:title"})
    if og_title and og_title.get("content"):
        texts.append(normalize_ws(og_title.get("content")))

    if race_name:
        texts.append(race_name)

    joined = " | ".join(texts)
    grade_name = normalize_grade_name(joined)
    return GRADE_NAME_TO_CODE.get(grade_name, "")


def extract_race_name_from_shutuba(soup: BeautifulSoup) -> str:
    candidates = []

    for sel in [
        "h1.RaceName",
        "div.RaceName h1",
        "div.RaceName",
        "h1",
    ]:
        tag = soup.select_one(sel)
        if tag:
            txt = normalize_ws(tag.get_text(" ", strip=True))
            if txt:
                candidates.append(txt)

    if soup.title and normalize_ws(soup.title.get_text()):
        candidates.append(normalize_ws(soup.title.get_text()))

    for text in candidates:
        s = clean_race_name_text(text)
        if s and s not in ("出馬表",):
            return s

    return "レース名不明"


def extract_date_from_href_like_text(text: str) -> Optional[date]:
    """
    地方競馬ページ内のURLやHTML断片から開催日を拾う。

    例:
    - kaisai_date=20260505
    - kaisai_id=2026430505
      地方の kaisai_id は YYYY + 場コード2桁 + MMDD の形。
    """
    if not text:
        return None

    s = str(text)

    # 最優先: kaisai_date=YYYYMMDD
    for m in re.finditer(r"(?:kaisai_date|kaisaiDate)=([0-9]{8})", s):
        d = date_from_yyyymmdd(m.group(1))
        if d is not None:
            return d

    # 地方: kaisai_id=YYYY + 場コード2桁 + MMDD
    # 例: 2026430505 -> 2026/05/05
    for m in re.finditer(r"(?:kaisai_id|kaisaiId)=([0-9]{10})", s):
        kaisai_id = m.group(1)
        yyyymmdd = kaisai_id[:4] + kaisai_id[6:10]
        d = date_from_yyyymmdd(yyyymmdd)
        if d is not None:
            return d

    return None


def infer_nar_race_date_from_race_id(race_id: str) -> Optional[date]:
    """
    NAR race_id から開催日を復元する。

    地方 netkeiba の race_id は概ね:
    YYYY + 場コード2桁 + MMDD + レース番号2桁

    例:
    202643050511
    -> 2026 + 43 + 0505 + 11
    -> 2026/05/05
    """
    if not race_id:
        return None

    s = str(race_id).strip()
    if not re.fullmatch(r"\d{12}", s):
        return None

    if not is_nar_race_id(s):
        return None

    yyyymmdd = s[:4] + s[6:10]
    return date_from_yyyymmdd(yyyymmdd)


def extract_race_id_from_shutuba_soup(soup: BeautifulSoup) -> str:
    """
    shutuba HTML 内から race_id を拾う。
    head が欠けたキャッシュ断片でも拾えるように HTML 全体も見る。
    """
    html = str(soup)

    patterns = [
        r"race_id=(\d{12})",
        r"""_shutuba_race_id\s*=\s*['"](\d{12})['"]""",
        r"""name=['"]race_id['"][^>]+value=['"](\d{12})['"]""",
        r"""value=['"](\d{12})['"][^>]+name=['"]race_id['"]""",
        r"MyRace-[A-Za-z_]*-?Item-?(\d{12})",
        r"MyRace-Item-(\d{12})",
        r"horse_(\d{12})",
    ]

    for p in patterns:
        m = re.search(p, html)
        if m:
            return m.group(1)

    return ""


def extract_race_date_from_shutuba(soup: BeautifulSoup) -> str:
    """
    出馬表HTMLから対象レース日を取得する。

    修正ポイント:
    - EUC-JP文字化けが残っていても、race_id から地方開催日を復元する
    - 地方ページの日付ナビの「前」「次」を誤って拾わない
    - meta/title の通常日付表現を優先し、汎用リンク走査は最後に回す
    """
    primary_candidates = []

    # 1) ページ自身を説明する meta/title/レース名周辺を優先
    meta_desc = soup.find("meta", attrs={"name": "description"})
    if meta_desc and meta_desc.get("content"):
        primary_candidates.append(meta_desc.get("content"))

    og_title = soup.find("meta", attrs={"property": "og:title"})
    if og_title and og_title.get("content"):
        primary_candidates.append(og_title.get("content"))

    og_desc = soup.find("meta", attrs={"property": "og:description"})
    if og_desc and og_desc.get("content"):
        primary_candidates.append(og_desc.get("content"))

    if soup.title and soup.title.get_text():
        primary_candidates.append(soup.title.get_text())

    h1 = (
        soup.select_one("h1.RaceName")
        or soup.select_one("div.RaceName h1")
        or soup.select_one("h1")
    )
    if h1 and h1.parent:
        primary_candidates.append(h1.parent.get_text(" ", strip=True))

    for text in primary_candidates:
        d = parse_date_any(text)
        if d is not None:
            return date_to_ymd_slash(d)

    # 2) 地方は race_id から YYYY + 場コード + MMDD + RR を復元できる。
    #    日付ナビの「前」「次」リンクよりもこちらを優先する。
    race_id = extract_race_id_from_shutuba_soup(soup)
    d = infer_nar_race_date_from_race_id(race_id)
    if d is not None:
        return date_to_ymd_slash(d)

    # 3) Active な日付ナビがある場合のみ拾う。これなら「前」「次」を避けられる。
    active_date_links = []
    for sel in [
        "#RaceList_DateList dd.Active a[href]",
        ".RaceList_Date dd.Active a[href]",
        "dd.Active a[href*='kaisai_date=']",
    ]:
        for a in soup.select(sel):
            href = a.get("href", "")
            txt = a.get_text(" ", strip=True)
            if href:
                active_date_links.append(href)
            if txt:
                active_date_links.append(txt)

    for text in active_date_links:
        d = parse_date_any(text)
        if d is not None:
            return date_to_ymd_slash(d)
        d = extract_date_from_href_like_text(text)
        if d is not None:
            return date_to_ymd_slash(d)

    # 4) canonical / og:url / alternate / 払戻リンクなど、ページ自身に近いURLだけを見る
    secondary_candidates = []

    og_url = soup.find("meta", attrs={"property": "og:url"})
    if og_url and og_url.get("content"):
        secondary_candidates.append(og_url.get("content"))

    for link in soup.select("link[rel='canonical'][href], link[rel='alternate'][href]"):
        href = link.get("href", "")
        if href:
            secondary_candidates.append(href)

    for a in soup.select("div.Refundlink a[href], a[href*='kaisai_date='][href*='payback_list']"):
        href = a.get("href", "")
        if href:
            secondary_candidates.append(href)

    for text in secondary_candidates:
        d = parse_date_any(text)
        if d is not None:
            return date_to_ymd_slash(d)
        d = extract_date_from_href_like_text(text)
        if d is not None:
            return date_to_ymd_slash(d)

    # 5) 最終手段: HTML全体。複数の kaisai_date がある場合は race_id と一致する日付を優先。
    html = str(soup)

    if race_id:
        d = infer_nar_race_date_from_race_id(race_id)
        if d is not None:
            target_yyyymmdd = d.strftime("%Y%m%d")
            if target_yyyymmdd in html:
                return date_to_ymd_slash(d)

    d = extract_date_from_href_like_text(html)
    if d is not None:
        return date_to_ymd_slash(d)

    return ""

def extract_course_info_from_shutuba(soup: BeautifulSoup) -> Dict[str, str]:
    out = {
        "競馬場名": "",
        "競馬場コード": "",
        "距離": "",
        "距離数値": "",
        "コース種別": "",
    }

    race_data_01 = soup.select_one("div.RaceData01")
    text01 = normalize_ws(race_data_01.get_text(" ", strip=True)) if race_data_01 else ""

    m = re.search(r"(芝|ダート|ダ|障害|障)\s*(\d{3,4})m", text01)
    if m:
        course_type = m.group(1)
        distance_num = m.group(2)

        if course_type == "ダート":
            course_type = "ダ"
        elif course_type == "障害":
            course_type = "障"

        out["コース種別"] = course_type
        out["距離数値"] = distance_num
        out["距離"] = f"{course_type}{distance_num}m"

    race_data_02 = soup.select_one("div.RaceData02")
    if race_data_02:
        spans = [normalize_ws(span.get_text(" ", strip=True)) for span in race_data_02.select("span")]
        for s in spans:
            if s in JYO_NAME_TO_CODE:
                out["競馬場名"] = s
                out["競馬場コード"] = JYO_NAME_TO_CODE[s]
                break

    if not out["競馬場コード"]:
        course_link = soup.select_one('a.Popup_Img.MapRaceBtn[href*="/img/course/"]')
        href = course_link.get("href", "") if course_link else ""
        m = re.search(r"/img/course/(\d{2})_", href)
        if m:
            code = m.group(1)
            out["競馬場コード"] = code
            for name, c in JYO_NAME_TO_CODE.items():
                if c == code:
                    out["競馬場名"] = name
                    break

    if not out["競馬場コード"]:
        refund_link = soup.select_one("div.Refundlink a")
        refund_text = normalize_ws(refund_link.get_text(" ", strip=True)) if refund_link else ""
        for name, code in JYO_NAME_TO_CODE.items():
            if name in refund_text:
                out["競馬場名"] = name
                out["競馬場コード"] = code
                break

    return out


def extract_horse_id(horse_url: str) -> str:
    match = re.search(r'/horse/(\d+)', horse_url)
    return match.group(1) if match else ""


def is_nar_race(race_url: str) -> bool:
    return "nar.netkeiba.com" in race_url


def get_horse_rows(soup: BeautifulSoup, is_nar: bool):
    if is_nar:
        return soup.select("table.RaceTable01.ShutubaTable tr.HorseList")
    return soup.select("table.Shutuba_Table tr.HorseList")


def extract_weight(row, nar_flag: bool) -> str:
    if nar_flag:
        tds = row.find_all("td")
        return normalize_ws(tds[5].get_text(" ", strip=True)) if len(tds) > 5 else ""

    barei_td = row.find("td", class_="Barei")
    weight_td = barei_td.find_next_sibling("td") if barei_td else None
    return normalize_ws(weight_td.get_text(" ", strip=True)) if weight_td else ""


def parse_corner_passage(soup: BeautifulSoup) -> Dict[str, str]:
    table = soup.find("table", summary="コーナー通過順位")
    if table is None:
        table = soup.select_one("table.Corner_Num")
    if table is None:
        return {}

    out = {}
    for tr in table.select("tr"):
        th = tr.find("th")
        td = tr.find("td")
        if th is None or td is None:
            continue

        key = normalize_ws(th.get_text(" ", strip=True)).replace(" ", "")
        val = normalize_ws(td.get_text(" ", strip=True))
        if key:
            out[key] = val

    return out


def parse_lap_and_pace(soup: BeautifulSoup) -> Dict[str, object]:
    result = {
        "ペース": "",
        "ラップタイム_累計": [],
        "ラップタイム_ハロンごと": [],
    }

    pace_box = soup.select_one("div.Contents_Header.RapPace div.RapPace_Title")
    if pace_box:
        pace_text = normalize_ws(pace_box.get_text(" ", strip=True))
        pace_text = pace_text.replace("ペース:", "").replace("ペース", "").replace(":", "").strip()
        result["ペース"] = pace_text

    lap_table = soup.find("table", summary="ラップタイム")
    if lap_table is None:
        lap_table = soup.select_one("table.Race_HaronTime")

    if lap_table is None:
        return result

    haron_rows = lap_table.select("tr.HaronTime")

    if len(haron_rows) >= 1:
        result["ラップタイム_累計"] = [
            normalize_ws(td.get_text(" ", strip=True)) for td in haron_rows[0].find_all("td")
        ]

    if len(haron_rows) >= 2:
        result["ラップタイム_ハロンごと"] = [
            normalize_ws(td.get_text(" ", strip=True)) for td in haron_rows[1].find_all("td")
        ]

    return result


def fetch_race_results_dynamic(horse_id: str) -> List[Dict]:
    """
    競走馬の成績一覧を取得する。

    重要:
    horse/result ページは、直近で別レースに出走していると内容が更新される。
    そのため FORCE_REFRESH_HORSE_RESULT_PAGES=True のときはキャッシュを読まず、
    毎回ネットワークから取り直す。
    取得後のHTMLは CACHE_HORSE_RESULT_PAGES_AFTER_REFRESH=True ならキャッシュへ保存する。
    """
    if not horse_id:
        return []

    url = make_horse_result_url(horse_id)
    soup = fetch_soup(
        url,
        force_refresh=FORCE_REFRESH_HORSE_RESULT_PAGES,
        read_cache=not FORCE_REFRESH_HORSE_RESULT_PAGES,
        write_cache=CACHE_HORSE_RESULT_PAGES_AFTER_REFRESH,
    )

    table = soup.select_one("table.db_h_race_results")
    if table is None:
        return []

    header_ths = table.select("thead th")
    headers_text = [normalize_ws(th.get_text(" ", strip=True)).replace("\n", "") for th in header_ths]
    col_index = {name: i for i, name in enumerate(headers_text)}

    results = []
    rows = table.select("tbody tr")

    for row in rows:
        cols = row.find_all("td")
        if len(cols) != len(headers_text):
            continue

        result = {}
        for key, idx in col_index.items():
            if not key or key in REMOVE_KEYS_PER_RECORD:
                continue

            if key == "レース名":
                link = cols[idx].find("a")
                result["race_url"] = link["href"] if link else ""

            text = normalize_ws(cols[idx].get_text(" ", strip=True))
            result[key] = text

        results.append(result)

    return results

def calculate_best_times_as_fields(race_results: List[Dict]) -> Dict:
    best = {}

    for r in race_results:
        distance = r.get("距離", "")
        time_str = r.get("タイム", "")
        if not distance or not time_str:
            continue

        sec = time_to_seconds(time_str)
        if sec is None:
            continue

        if distance not in best or sec < best[distance]:
            best[distance] = sec

    out = {}
    for dist, sec in best.items():
        out[f"持ちタイム_{dist}"] = seconds_to_time(sec)
    return out


def remove_records_on_or_after_target_date(race_results: List[Dict], target_race_date: str) -> List[Dict]:
    if not race_results:
        return []

    target_date = parse_date_any(target_race_date)
    if target_date is None:
        return race_results

    cleaned = []
    for r in race_results:
        rec_date = parse_date_any(r.get("日付"))
        if rec_date is None or rec_date < target_date:
            cleaned.append(r)

    return cleaned


def build_race_json(race_url: str, soup: BeautifulSoup) -> Dict:
    course_info = extract_course_info_from_shutuba(soup)
    raw_race_name = extract_race_name_from_shutuba(soup)

    grade_code = extract_grade_code_from_target_race(soup, raw_race_name)
    grade_name = ""
    for k, v in GRADE_NAME_TO_CODE.items():
        if v == grade_code:
            grade_name = k
            break

    race_json = {
        "race_id": extract_race_id_from_url(race_url),
        "race_url": race_url,
        "is_nar": False,
        "race_name": raw_race_name,
        "race_name_search": normalize_race_name_for_search(raw_race_name),
        "race_date": "",
        "競馬場名": course_info.get("競馬場名", ""),
        "競馬場コード": course_info.get("競馬場コード", ""),
        "距離": course_info.get("距離", ""),
        "距離数値": course_info.get("距離数値", ""),
        "コース種別": course_info.get("コース種別", ""),
        "grade_name": grade_name,
        "grade_code": grade_code,
        "horses": []
    }

    race_json["race_date"] = extract_race_date_from_shutuba(soup)

    nar_flag = is_nar_race(race_url)
    race_json["is_nar"] = nar_flag

    rows = get_horse_rows(soup, nar_flag)

    for row in rows:
        horse_tag = row.select_one("span.HorseName a")
        if horse_tag is None:
            continue

        waku_td = row.find("td", class_=re.compile("^Waku"))
        umaban_td = row.find("td", class_=re.compile("^Umaban"))
        if not waku_td or not umaban_td:
            continue

        horse_name = normalize_ws(horse_tag.get_text(" ", strip=True))
        horse_url = horse_tag.get("href", "")
        horse_id = extract_horse_id(horse_url)
        weight = extract_weight(row, nar_flag)

        jockey_tag = row.select_one("td.Jockey a")
        jockey = normalize_ws(jockey_tag.get_text(" ", strip=True)) if jockey_tag else ""

        race_json["horses"].append({
            "枠": normalize_ws(waku_td.get_text(" ", strip=True)),
            "馬番": normalize_ws(umaban_td.get_text(" ", strip=True)),
            "馬名": horse_name,
            "斤量": weight,
            "騎手": jockey,
            "馬ID": horse_id,
            "馬URL": horse_url,
            "持ちタイム": {},
            "過去成績": []
        })

    return race_json


def build_matchup_json_sorted_filtered(racedata: Dict) -> Dict:
    starters = {h.get("馬名", "") for h in racedata.get("horses", []) if h.get("馬名")}
    races_map = {}

    for h in racedata.get("horses", []):
        name = h.get("馬名", "")
        if not name:
            continue

        for r in h.get("過去成績", []):
            race_url = r.get("race_url")
            if not race_url:
                continue

            bucket = races_map.setdefault(race_url, {
                "race_url": race_url,
                "日付": r.get("日付"),
                "開催": r.get("開催"),
                "R": r.get("R"),
                "レース名": r.get("レース名"),
                "距離": r.get("距離"),
                "馬場": r.get("馬場"),
                "頭数": r.get("頭数"),
                "participants": {}
            })

            if name in starters:
                bucket["participants"][name] = {
                    "順位": r.get("着順"),
                    "タイム": r.get("タイム")
                }

    races_list = []

    for race in races_map.values():
        if len(race["participants"]) <= 1:
            continue

        items = []
        for horse_name, info in race["participants"].items():
            rank_int = safe_int(info.get("順位"))
            sort_rank = rank_int if rank_int is not None else 10**9
            items.append((horse_name, info, sort_rank))

        items.sort(key=lambda x: x[2])

        sorted_participants = {}
        for horse_name, info, _ in items:
            rank_int = safe_int(info.get("順位"))
            sorted_participants[horse_name] = {
                "順位": rank_int if rank_int is not None else info.get("順位"),
                "タイム": info.get("タイム")
            }

        race["participants"] = sorted_participants
        races_list.append(race)

    races_list.sort(key=lambda r: normalize_date_for_sort(r.get("日付")), reverse=True)

    return {
        "target_race_url": racedata.get("race_url", ""),
        "target_race_date": racedata.get("race_date", ""),
        "races": races_list
    }


def build_racedata_jsonl_records(racedata: Dict) -> List[Dict]:
    records = []
    for horse in racedata.get("horses", []):
        records.append({
            "target_race_url": racedata.get("race_url", ""),
            "target_race_name": racedata.get("race_name", ""),
            "target_race_date": racedata.get("race_date", ""),
            "target_競馬場名": racedata.get("競馬場名", ""),
            "target_競馬場コード": racedata.get("競馬場コード", ""),
            "target_距離": racedata.get("距離", ""),
            "target_距離数値": racedata.get("距離数値", ""),
            "target_コース種別": racedata.get("コース種別", ""),
            "target_格名": racedata.get("grade_name", ""),
            "target_格コード": racedata.get("grade_code", ""),
            "is_nar": racedata.get("is_nar", False),
            **horse
        })
    return records


def build_matchup_jsonl_records(matchup: Dict) -> List[Dict]:
    records = []
    for race in matchup.get("races", []):
        records.append({
            "target_race_url": matchup.get("target_race_url", ""),
            "target_race_date": matchup.get("target_race_date", ""),
            **race
        })
    return records


def sanitize_header(s: str) -> str:
    if s is None:
        return ""
    s = str(s).replace("\n", " ").replace("\r", " ").strip()
    s = s.replace("\u00a0", " ")
    s = re.sub(r"\s+", " ", s)
    return s


def normalize_horse_name(name: str) -> str:
    if name is None:
        return ""
    s = str(name).strip().replace("\u00a0", " ")
    s = s.replace(" ", "").replace("　", "")
    s = re.sub(r"\s+", "", s)
    return s


def distance_label(dist: str) -> str:
    d = sanitize_header(dist)
    if not d:
        return ""
    if d.endswith("m"):
        return d
    if re.search(r"\d+$", d):
        return d + "m"
    return d


def build_horses_sorted_by_umaban(racedata: Dict):
    horses = racedata.get("horses", [])
    enriched = []
    for h in horses:
        name = h.get("馬名", "")
        umaban = h.get("馬番", "")
        umaban_i = to_int_safe(umaban)
        enriched.append((umaban_i if umaban_i is not None else 10**9, str(umaban), str(name)))
    enriched.sort(key=lambda x: x[0])
    return [(umaban_str, name) for _, umaban_str, name in enriched if name and name.strip()]


def build_unique_headers(races_list: List[Dict]):
    base_names = []
    for r in races_list:
        name = sanitize_header(r.get("レース名", ""))
        dist = distance_label(r.get("距離", ""))
        base_names.append(f"{name}（{dist}）" if dist else f"{name}")

    cnt = Counter(base_names)

    headers = []
    for r, base in zip(races_list, base_names):
        if cnt[base] >= 2:
            date_str = sanitize_header(r.get("日付", ""))
            if "（" in base and base.endswith("）"):
                base_no_paren = base[:-1]
                headers.append(f"{base_no_paren}・{date_str}）" if date_str else f"{base_no_paren}・日付不明）")
            else:
                headers.append(f"{base}（{date_str}）" if date_str else f"{base}（日付不明）")
        else:
            headers.append(base)

    return headers


def cell_text(rank, time_str):
    r = "" if rank is None else str(rank)
    t = "" if time_str is None else str(time_str)
    if r == "" and t == "":
        return ""
    if t == "":
        return f"{r}"
    if r == "":
        return f"({t})"
    return f"{r}({t})"


def write_matchup_csv(matchup: Dict, racedata: Dict, output_csv_path: str):
    races_list = matchup.get("races", [])
    if isinstance(races_list, dict):
        races_list = list(races_list.values())

    horses_sorted = build_horses_sorted_by_umaban(racedata)
    headers = build_unique_headers(races_list)

    race_participants_norm = []
    for r in races_list:
        p = r.get("participants", {}) or {}
        norm_map = {}
        for k, v in p.items():
            norm_k = normalize_horse_name(k)
            if norm_k:
                norm_map[norm_k] = v
        race_participants_norm.append(norm_map)

    with open(output_csv_path, "w", encoding="utf-8-sig", newline="") as f:
        w = csv.writer(f)
        w.writerow(["馬番", "馬名", *headers])

        for umaban, horse_name in horses_sorted:
            norm_name = normalize_horse_name(horse_name)
            row = [umaban, horse_name]

            for p_norm in race_participants_norm:
                info = p_norm.get(norm_name)
                if not info:
                    row.append("")
                    continue
                row.append(cell_text(info.get("順位"), info.get("タイム")))

            w.writerow(row)


def quote_netkeiba_word(text: str) -> str:
    return quote(text, safe="", encoding="euc_jp", errors="strict")


def extract_race_id_from_url(url: str) -> str:
    if not url:
        return ""

    m = re.search(r"/race/(\d{12})/?", url)
    if m:
        return m.group(1)

    m = re.search(r"race_id=(\d{12})", url)
    if m:
        return m.group(1)

    return ""


def build_race_list_search_url(
    search_word: Optional[str] = None,
    jyo_list: Optional[List[str]] = None,
    kyori_list: Optional[List[str]] = None,
    grade_list: Optional[List[str]] = None,
) -> str:
    parts = ["pid=race_list"]

    if search_word:
        parts.append(f"word={quote_netkeiba_word(search_word)}")
    else:
        parts.append("word=")

    if jyo_list:
        for j in jyo_list:
            parts.append(f"jyo%5B%5D={str(j)}")

    parts.append("kyori_min=")
    parts.append("kyori_max=")

    if kyori_list:
        for k in kyori_list:
            parts.append(f"kyori%5B%5D={str(k)}")

    if grade_list:
        for g in grade_list:
            parts.append(f"grade%5B%5D={str(g)}")

    parts.append("sort=date")
    parts.append("list=20")

    return "https://db.netkeiba.com/?" + "&".join(parts)


def search_past_race_urls(
    search_word: Optional[str],
    target_race_date: str,
    limit: int = 10,
    jyo_list: Optional[List[str]] = None,
    kyori_list: Optional[List[str]] = None,
    grade_list: Optional[List[str]] = None,
) -> List[Dict]:
    search_url = build_race_list_search_url(
        search_word=search_word,
        jyo_list=jyo_list,
        kyori_list=kyori_list,
        grade_list=grade_list,
    )
    soup = fetch_soup(search_url)

    target_date_obj = parse_date_any(target_race_date)
    if target_date_obj is None:
        raise ValueError(f"対象レース日を解釈できません: {target_race_date}")

    items = []
    rows = soup.select("tr")
    seen_race_ids = set()

    for tr in rows:
        race_link = tr.select_one('td.w_race a[href*="/race/"]')
        date_link = tr.select_one('td a[href*="/race/list/"]')

        if race_link is None or date_link is None:
            continue

        row_race_name = race_link.get("title") or race_link.get_text(strip=True)
        row_race_name = normalize_ws(row_race_name)

        row_date_text = normalize_ws(date_link.get_text(strip=True))
        row_date_obj = parse_date_any(row_date_text)
        if row_date_obj is None or row_date_obj >= target_date_obj:
            continue

        db_race_url = race_link.get("href", "")
        race_id = extract_race_id_from_url(db_race_url)
        if not race_id or race_id in seen_race_ids:
            continue

        seen_race_ids.add(race_id)

        nar_flag = is_nar_race_id(race_id)

        items.append({
            "race_id": race_id,
            "race_name": row_race_name,
            "race_date": date_to_ymd_slash(row_date_obj),
            "result_url": build_result_url_from_race_id(race_id, nar_flag),
        })

    items.sort(key=lambda x: parse_date_any(x["race_date"]), reverse=True)
    return items[:limit]


def search_past_races_with_fallback(race_json: Dict, limit: int = 10):
    """
    方針:
    - レース名 + 開催場 + 距離 だけで検索する
    - 足りなくても別条件には広げない
    - 取れた分だけ返す
    """
    jyo_list = [race_json["競馬場コード"]] if race_json.get("競馬場コード") else None
    kyori_list = [race_json["距離数値"]] if race_json.get("距離数値") else None

    if PAST_RACE_SEARCH_WORD:
        search_words = [PAST_RACE_SEARCH_WORD]
    else:
        search_words = build_search_word_candidates(race_json.get("race_name", ""))

    collected = []
    seen_ids = set()
    logs = []

    if not jyo_list or not kyori_list:
        logs.append({
            "strategy": "stop",
            "reason": "競馬場コードまたは距離数値が取得できないため、name+jyo+kyori検索を実行できない",
            "collected": 0,
            "requested": limit,
        })
        return collected, logs

    for word in search_words:
        if len(collected) >= limit:
            break

        try:
            items = search_past_race_urls(
                search_word=word,
                target_race_date=race_json["race_date"],
                limit=limit,
                jyo_list=jyo_list,
                kyori_list=kyori_list,
                grade_list=None,
            )
        except Exception as e:
            logs.append({
                "strategy": "name+jyo+kyori",
                "word": word,
                "error": str(e),
            })
            continue

        added = 0
        for item in items:
            race_id = item.get("race_id")
            if race_id and race_id not in seen_ids:
                seen_ids.add(race_id)
                collected.append(item)
                added += 1
                if len(collected) >= limit:
                    break

        logs.append({
            "strategy": "name+jyo+kyori",
            "word": word,
            "hits": len(items),
            "added": added,
        })

    if len(collected) < limit:
        logs.append({
            "strategy": "stop",
            "reason": "name+jyo+kyori のみで検索し、他条件には広げない方針",
            "collected": len(collected),
            "requested": limit,
        })

    collected.sort(key=lambda x: parse_date_any(x["race_date"]), reverse=True)
    return collected[:limit], logs


def get_text_or_empty(tag) -> str:
    if tag is None:
        return ""
    return normalize_ws(tag.get_text(" ", strip=True))


def normalize_result_header(text: str) -> str:
    s = normalize_ws(text)
    s = s.replace(" ", "")
    s = s.replace("\n", "")
    s = s.replace("\r", "")
    return s


def parse_result_table(soup: BeautifulSoup) -> List[Dict]:
    table = soup.select_one("#All_Result_Table")
    if table is None:
        return []

    header_ths = table.select("thead th")
    headers = [normalize_result_header(th.get_text(" ", strip=True)) for th in header_ths]
    headers = [h for h in headers if h]
    if not headers:
        return []

    results = []

    for tr in table.select("tbody tr"):
        tds = tr.find_all("td", recursive=False)
        if not tds:
            continue

        row_map = {}

        for header, td in zip(headers, tds):
            val = get_text_or_empty(td)

            if header == "着順":
                val = get_text_or_empty(td.select_one(".Rank")) or val
            elif header == "馬名":
                a = td.select_one("a[title]") or td.select_one("a")
                if a is not None:
                    val = normalize_ws(a.get("title") or a.get_text(" ", strip=True))
            elif header == "性齢":
                val = get_text_or_empty(td.select_one(".Detail_Left")) or val
            elif header == "斤量":
                val = get_text_or_empty(td.select_one(".JockeyWeight")) or val
            elif header in ("タイム", "着差"):
                val = get_text_or_empty(td.select_one(".RaceTime")) or val
            elif header == "人気":
                val = get_text_or_empty(td.select_one(".OddsPeople")) or val
            elif header == "単勝オッズ":
                val = get_text_or_empty(td.select_one(".Odds_Ninki")) or val

            row_map[header] = val

        record = {
            "着順": row_map.get("着順", ""),
            "枠": row_map.get("枠", ""),
            "馬番": row_map.get("馬番", ""),
            "馬名": row_map.get("馬名", ""),
            "性齢": row_map.get("性齢", ""),
            "斤量": row_map.get("斤量", ""),
            "騎手": row_map.get("騎手", ""),
            "タイム": row_map.get("タイム", ""),
            "着差": row_map.get("着差", ""),
            "人気": row_map.get("人気", ""),
            "単勝オッズ": row_map.get("単勝オッズ", ""),
            "後3F": row_map.get("後3F", ""),
            "コーナー通過順": row_map.get("コーナー通過順", ""),
            "厩舎": row_map.get("厩舎", ""),
            "馬体重(増減)": row_map.get("馬体重(増減)", ""),
        }

        results.append(record)

    return results


def fetch_one_past_race(item: Dict, target_race_url: str, target_race_name: str, target_race_date: str) -> Dict:
    soup = fetch_soup(item["result_url"])
    results = parse_result_table(soup)
    corner_passage = parse_corner_passage(soup)
    lap_info = parse_lap_and_pace(soup)

    return {
        "target_race_url": target_race_url,
        "target_race_name": target_race_name,
        "target_race_date": target_race_date,
        "past_race_id": item["race_id"],
        "past_race_name": item["race_name"],
        "past_race_date": item["race_date"],
        "past_race_result_url": item["result_url"],
        "結果": results,
        "コーナー通過順位": corner_passage,
        "ペース": lap_info["ペース"],
        "ラップタイム_累計": lap_info["ラップタイム_累計"],
        "ラップタイム_ハロンごと": lap_info["ラップタイム_ハロンごと"],
    }


def drop_empty_values(obj):
    if isinstance(obj, dict):
        new_obj = {}
        for k, v in obj.items():
            cleaned = drop_empty_values(v)
            if cleaned in ("", None, [], {}):
                continue
            new_obj[k] = cleaned
        return new_obj

    if isinstance(obj, list):
        new_list = []
        for v in obj:
            cleaned = drop_empty_values(v)
            if cleaned in ("", None, [], {}):
                continue
            new_list.append(cleaned)
        return new_list

    return obj


def clean_result_row(row: Dict) -> Dict:
    keep_keys = [
        "着順", "枠", "馬番", "馬名", "性齢", "斤量", "騎手",
        "タイム", "着差", "人気", "単勝オッズ", "後3F",
        "コーナー通過順", "厩舎", "馬体重(増減)",
    ]
    cleaned = {k: row.get(k, "") for k in keep_keys if k in row}
    return drop_empty_values(cleaned)


def clean_record_for_jsonl(record: Dict) -> Dict:
    cleaned = {
        "target_race_name": record.get("target_race_name", ""),
        "target_race_date": record.get("target_race_date", ""),
        "past_race_id": record.get("past_race_id", ""),
        "past_race_name": record.get("past_race_name", ""),
        "past_race_date": record.get("past_race_date", ""),
        "結果": [clean_result_row(r) for r in record.get("結果", [])],
        "コーナー通過順位": record.get("コーナー通過順位", {}),
        "ペース": record.get("ペース", ""),
        "ラップタイム_累計": record.get("ラップタイム_累計", []),
        "ラップタイム_ハロンごと": record.get("ラップタイム_ハロンごと", []),
    }
    return drop_empty_values(cleaned)


def clean_records_for_jsonl(records: List[Dict]) -> List[Dict]:
    return [clean_record_for_jsonl(r) for r in records]


def process_one_race(race_id: str):
    race_url = build_shutuba_url_from_race_id(race_id)

    # 各レースで最初にアクセスする入口ページは、古い出馬表キャッシュを避けるため
    # キャッシュを読まずにネットワークから取得する。取得後は通常どおり保存する。
    shutuba_soup = fetch_soup(
        race_url,
        force_refresh=FORCE_REFRESH_ENTRY_PAGE_PER_RACE,
        read_cache=not FORCE_REFRESH_ENTRY_PAGE_PER_RACE,
        write_cache=CACHE_ENTRY_PAGE_AFTER_REFRESH,
    )

    race_json = build_race_json(race_url, shutuba_soup)

    if not race_json["race_name"] or race_json["race_name"] == "レース名不明":
        raise ValueError(f"対象レース名を取得できませんでした: race_id={race_id}")

    if not race_json["race_date"]:
        detected_race_id = extract_race_id_from_shutuba_soup(shutuba_soup)
        inferred_date = date_to_ymd_slash(infer_nar_race_date_from_race_id(detected_race_id))

        print("[DEBUG] race_date empty")
        print("[DEBUG] detected_race_id:", detected_race_id)
        print("[DEBUG] inferred_date_from_race_id:", inferred_date)
        print("[DEBUG] title:", normalize_ws(shutuba_soup.title.get_text()) if shutuba_soup.title else "")

        raise ValueError(f"対象レース日を取得できませんでした: race_id={race_id}")

    if not race_json.get("horses"):
        raise ValueError(f"出走馬を取得できませんでした: race_id={race_id}")

    print("対象race_id:", race_id)
    print("対象レース名:", race_json["race_name"])
    print("検索用レース名:", race_json.get("race_name_search", ""))
    print("検索候補:", build_search_word_candidates(race_json["race_name"]))
    print("対象レース日:", race_json["race_date"])
    print("競馬場:", race_json.get("競馬場名", ""))
    print("競馬場コード:", race_json.get("競馬場コード", ""))
    print("距離:", race_json.get("距離", ""))
    print("距離数値:", race_json.get("距離数値", ""))
    print("格:", race_json.get("grade_name", ""))
    print("格コード:", race_json.get("grade_code", ""))
    print("出走頭数:", len(race_json["horses"]))

    for horse in race_json["horses"]:
        print(f"Fetching horse results: {horse['馬名']}...")
        horse_id = horse["馬ID"]
        try:
            race_results_all = fetch_race_results_dynamic(horse_id)
            horse["過去成績"] = race_results_all
        except Exception as e:
            print("ERROR horse:", horse["馬名"], e)
            horse["過去成績"] = []

    enrich_horses_with_pedigree(race_json)

    for horse in race_json["horses"]:
        before_n = len(horse.get("過去成績", []))

        horse["過去成績"] = remove_records_on_or_after_target_date(
            horse.get("過去成績", []),
            race_json.get("race_date", "")
        )

        after_n = len(horse.get("過去成績", []))

        horse["持ちタイム"] = calculate_best_times_as_fields(
            horse.get("過去成績", [])
        )

        if before_n != after_n:
            print(f"[cleanup] {horse['馬名']}: {before_n} -> {after_n}")

    matchup_json = build_matchup_json_sorted_filtered(race_json)

    search_word = PAST_RACE_SEARCH_WORD or race_json.get("race_name_search") or race_json["race_name"]
    print("過去傾向検索語(初期):", search_word)

    past_races, search_logs = search_past_races_with_fallback(
        race_json=race_json,
        limit=PAST_RACE_LIMIT,
    )

    print("検索ログ:")
    for log in search_logs:
        print(log)

    print("取得した過去レース件数:", len(past_races))
    for i, r in enumerate(past_races, 1):
        print(f"{i:02d}. {r['race_date']} {r['race_name']} {r['race_id']}")

    trend_records = []
    for i, item in enumerate(past_races, 1):
        print(f"[{i}/{len(past_races)}] fetching trend: {item['result_url']}")
        try:
            record = fetch_one_past_race(
                item=item,
                target_race_url=race_url,
                target_race_name=race_json["race_name"],
                target_race_date=race_json["race_date"]
            )
            trend_records.append(record)
        except Exception as e:
            print("ERROR trend:", item["result_url"], e)

    trend_records_for_save = clean_records_for_jsonl(trend_records)

    file_prefix = build_output_prefix(race_json)

    racedata_jsonl_path = os.path.join(OUTPUT_DIR, f"{file_prefix}_出馬表.jsonl")
    matchup_jsonl_path = os.path.join(OUTPUT_DIR, f"{file_prefix}_対戦表.jsonl")
    trend_jsonl_path = os.path.join(OUTPUT_DIR, f"{file_prefix}_過去10年傾向.jsonl")
    csv_path = os.path.join(OUTPUT_DIR, f"{file_prefix}_対戦表.csv")

    write_jsonl(build_racedata_jsonl_records(race_json), racedata_jsonl_path)
    print("saved:", racedata_jsonl_path)

    write_jsonl(build_matchup_jsonl_records(matchup_json), matchup_jsonl_path)
    print("saved:", matchup_jsonl_path)

    write_jsonl(trend_records_for_save, trend_jsonl_path)
    print("saved:", trend_jsonl_path)

    if WRITE_MATCHUP_CSV:
        write_matchup_csv(matchup_json, race_json, csv_path)
        print("saved:", csv_path)


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    if not RACE_ID_LIST:
        raise ValueError("RACE_ID_LIST が空です")

    prev = get_cache_metrics()

    for idx, race_id in enumerate(RACE_ID_LIST, 1):
        print("=" * 80)
        print(f"[{idx}/{len(RACE_ID_LIST)}] start race_id={race_id}")
        try:
            process_one_race(race_id)
            sync_all_cache_to_drive()
        except Exception as e:
            print(f"ERROR race_id={race_id}:", e)
            sync_all_cache_to_drive()

        now = get_cache_metrics()
        delta = {k: now.get(k, 0) - prev.get(k, 0) for k in set(now) | set(prev)}
        print(f"=== CACHE DELTA {race_id} ===")
        print("memory_hit   :", delta.get("memory_hit", 0))
        print("sqlite_hit   :", delta.get("sqlite_hit", 0))
        print("network_fetch:", delta.get("network_fetch", 0))
        print("cache_bypass :", delta.get("cache_bypass", 0))
        print("drive_restore:", delta.get("drive_restore", 0))
        print("sqlite_miss  :", delta.get("sqlite_miss", 0))
        print("mojibake_repair:", delta.get("mojibake_repair", 0))
        prev = now


if __name__ == "__main__":
    try:
        main()
    finally:
        sync_all_cache_to_drive()
        print_cache_metrics()


[1/2] start race_id=202603020211
対象race_id: 202603020211
対象レース名: ラジオNIKKEI賞
検索用レース名: ラジオNIKKEI賞
検索候補: ['ラジオNIKKEI賞']
対象レース日: 2026/06/28
競馬場: 福島
競馬場コード: 03
距離: 芝1800m
距離数値: 1800
格: G3
格コード: 3
出走頭数: 16
Fetching horse results: ルージュボヤージュ...
Fetching horse results: クカイリモク...
Fetching horse results: ジーネキング...
Fetching horse results: サイモンシャリオ...
Fetching horse results: リッツパーティー...
Fetching horse results: コルテオソレイユ...
Fetching horse results: ショウナンガルフ...
Fetching horse results: ディールメーカー...
Fetching horse results: キンググローリー...
Fetching horse results: ガリレア...
Fetching horse results: コロナドブリッジ...
Fetching horse results: ローベルクランツ...
Fetching horse results: サノノグレーター...
Fetching horse results: スカイスプレンダー...
Fetching horse results: バドリナート...
Fetching horse results: スペルーチェ...
Fetching horse pedigree: ルージュボヤージュ...
Fetching horse pedigree: クカイリモク...
Fetching horse pedigree: ジーネキング...
Fetching horse pedigree: サイモンシャリオ...
Fetching horse pedigree: リッツパーティー...
Fetching horse pedigree: コルテオソレイユ...
Fetching horse 